
***train.ipynb***
--------
Load the landmark archive produced by extract_landmarks.ipynb, build a fully-
connected classifier, train it, print a classification report, and export the
trained model together with metadata needed by the demo script.

Outputs
-------
  models/asl_model.keras        – saved Keras model
  models/label_classes.npy      – class name array  (shape: [26])
  models/training_history.npz   – accuracy / loss curves for optional plotting
  logs/training.log             – text log of every epoch

Usage:
    python src/train.ipynb


In [8]:
#imports

import os
import sys
import time
import csv
import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib
matplotlib.use("Agg") # non-interactive backend – safe on headless servers
import matplotlib.pyplot as plt
import seaborn as sns

print(f"PyTorch {torch.version} | CUDA: {torch.cuda.is_available()}")


PyTorch <module 'torch.version' from '/usr/local/lib/python3.11/site-packages/torch/version.py'> | CUDA: False


In [9]:
# Paths

BASE_DIR = os.path.join(os.path.dirname(os.path.abspath("file")), "..")
DATA_PATH = os.path.join(BASE_DIR, "data", "landmarks.npz")
MODEL_DIR = os.path.join(BASE_DIR, "models")
LOG_DIR = os.path.join(BASE_DIR, "logs")
MODEL_PATH = os.path.join(MODEL_DIR, "asl_model.pt")
CLASSES_PATH = os.path.join(MODEL_DIR, "label_classes.npy")
HISTORY_PATH = os.path.join(MODEL_DIR, "training_history.npz")
LOG_PATH = os.path.join(LOG_DIR, "training.log")

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)


In [10]:
# Hyper-parameters
VALIDATION_SPLIT = 0.20
RANDOM_SEED = 42
BATCH_SIZE = 128
MAX_EPOCHS = 100
PATIENCE = 10 # early-stopping patience
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4 # L2 regularisation


In [11]:
# 1. Load data
def load_data():
    if not os.path.exists(DATA_PATH):
        sys.exit(
        f"[ERROR] Landmark file not found: {DATA_PATH}\n"
"Run extract_landmarks.ipynb first."
)
    archive = np.load(DATA_PATH, allow_pickle=True)
    X = archive["X"].astype(np.float32)
    y = archive["y"].astype(np.int64)
    classes = archive["classes"] # e.g. ['A','B',...,'Z']
    print(f"Loaded {X.shape[0]:,} samples | feature dim={X.shape[1]} | classes={len(classes)}")
    return X, y, classes

In [12]:
# 2. Build model
class ASLClassifier(nn.Module):
    """
    Fully-connected classifier.
    Architecture: Dense(512) → BN → ReLU → Dropout →
                  Dense(256) → BN → ReLU → Dropout →
                  Dense(128) → BN → ReLU → Dropout →
                  Dense(num_classes)
    """

    def __init__(self, input_dim: int, num_classes: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(512, 256),       nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 128),       nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        return self.net(x)

In [13]:
# 3. Evaluation helpers
def plot_history(history):
    """Save accuracy and loss curves to models/training_curves.png."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(history["train_acc"], label="train")
    axes[0].plot(history["val_acc"],   label="val")
    axes[0].set_title("Accuracy")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()
    axes[0].grid(True)

    axes[1].plot(history["train_loss"], label="train")
    axes[1].plot(history["val_loss"],   label="val")
    axes[1].set_title("Loss")
    axes[1].set_xlabel("Epoch")
    axes[1].legend()
    axes[1].grid(True)

    fig.tight_layout()
    out = os.path.join(MODEL_DIR, "training_curves.png")
    fig.savefig(out, dpi=120)
    plt.close(fig)
    print(f"Saved training curves → {out}")

def plot_confusion(y_true, y_pred, classes):
    """Save normalised confusion matrix to models/confusion_matrix.png."""
    cm = confusion_matrix(y_true, y_pred, normalize="true")
    fig, ax = plt.subplots(figsize=(14, 12))
    sns.heatmap(
        cm, annot=True, fmt=".2f", cmap="Blues",
        xticklabels=classes, yticklabels=classes,
        ax=ax, annot_kws={"size": 7},
    )
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title("Confusion Matrix (normalised)")
    fig.tight_layout()
    out = os.path.join(MODEL_DIR, "confusion_matrix.png")
    fig.savefig(out, dpi=120)
    plt.close(fig)
    print(f"Saved confusion matrix → {out}")




In [ ]:
# 4. Main
def main():
    torch.manual_seed(RANDOM_SEED)
    np.random.seed(RANDOM_SEED)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # -- Load --
    X, y, classes = load_data()
    num_classes = len(classes)

    # -- Split --
    X_train, X_val, y_train, y_val = train_test_split(
        X, y,
        test_size=VALIDATION_SPLIT,
        random_state=RANDOM_SEED,
        stratify=y,
    )
    print(f"Train: {len(X_train):,}   Val: {len(X_val):,}")

    # -- DataLoaders --
    train_ds = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train))
    val_ds   = TensorDataset(torch.from_numpy(X_val),   torch.from_numpy(y_val))
    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
    val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    # -- Model --
    model     = ASLClassifier(X.shape[1], num_classes).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=5, min_lr=1e-6
    )
    print(model)

    # -- Training loop --
    history = {"train_acc": [], "val_acc": [], "train_loss": [], "val_loss": []}
    best_val_acc   = -1.0
    best_state     = None
    patience_count = 0

    log_rows = []
    t0 = time.time()

    for epoch in range(1, MAX_EPOCHS + 1):
        # --- train ---
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for xb, yb in train_dl:
            xb, yb = xb.to(device), yb.to(device)
            # Landmark noise augmentation — makes model robust to borderline detections
            xb = xb + torch.randn_like(xb) * 0.005
            optimizer.zero_grad()
            logits = model(xb)
            loss   = criterion(logits, yb)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * len(xb)
            correct      += (logits.argmax(1) == yb).sum().item()
            total        += len(xb)
        train_loss = running_loss / total
        train_acc  = correct / total

        # --- validate ---
        model.eval()
        val_loss_sum, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for xb, yb in val_dl:
                xb, yb = xb.to(device), yb.to(device)
                logits  = model(xb)
                loss    = criterion(logits, yb)
                val_loss_sum += loss.item() * len(xb)
                val_correct  += (logits.argmax(1) == yb).sum().item()
                val_total    += len(xb)
        val_loss = val_loss_sum / val_total
        val_acc  = val_correct  / val_total

        scheduler.step(val_loss)

        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        log_rows.append([epoch, train_loss, train_acc, val_loss, val_acc])

        print(f"Epoch {epoch:3d}/{MAX_EPOCHS}  "
              f"loss={train_loss:.4f}  acc={train_acc*100:.2f}%  "
              f"val_loss={val_loss:.4f}  val_acc={val_acc*100:.2f}%")

        # --- early stopping ---
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state   = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_count = 0
        else:
            patience_count += 1
            if patience_count >= PATIENCE:
                print(f"\nEarly stopping at epoch {epoch} (no improvement for {PATIENCE} epochs).")
                break

    elapsed = time.time() - t0
    print(f"\nTraining finished in {elapsed:.1f}s")

    # -- Restore best weights --
    model.load_state_dict(best_state)
    model.eval()

    # -- Final evaluation --
    def evaluate(dl):
        correct, total = 0, 0
        preds = []
        with torch.no_grad():
            for xb, yb in dl:
                xb, yb = xb.to(device), yb.to(device)
                logits = model(xb)
                pred   = logits.argmax(1)
                correct += (pred == yb).sum().item()
                total   += len(xb)
                preds.extend(pred.cpu().numpy())
        return correct / total, np.array(preds)

    train_acc, _      = evaluate(train_dl)
    val_acc,   y_pred = evaluate(val_dl)
    print(f"\nFinal train accuracy : {train_acc*100:.2f}%")
    print(f"Final val   accuracy : {val_acc*100:.2f}%")
    print("\n" + classification_report(y_val, y_pred, target_names=classes))

    # -- Save plots --
    plot_history(history)
    plot_confusion(y_val, y_pred, classes)

    # -- Write CSV log --
    with open(LOG_PATH, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["epoch", "loss", "accuracy", "val_loss", "val_accuracy"])
        writer.writerows(log_rows)
    print(f"Log saved → {LOG_PATH}")

    # -- Save history arrays --
    np.savez_compressed(
        HISTORY_PATH,
        train_acc =np.array(history["train_acc"]),
        val_acc   =np.array(history["val_acc"]),
        train_loss=np.array(history["train_loss"]),
        val_loss  =np.array(history["val_loss"]),
    )

    # -- Export model (checkpoint compatible with demo.py) --
    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "input_dim":        X.shape[1],
            "num_classes":      num_classes,
        },
        MODEL_PATH,
    )
    np.save(CLASSES_PATH, classes)
    print(f"\nModel saved → {MODEL_PATH}")
    print(f"Classes saved → {CLASSES_PATH}")

    if val_acc >= 0.85:
        print(f"\n[PASS] Validation accuracy {val_acc*100:.2f}% >= 85% target.")
    else:
        print(f"\n[NOTE] Validation accuracy {val_acc*100:.2f}% < 85% — consider more epochs or tuning.")


main()

Using device: cpu
Loaded 75,290 samples | feature dim=63 | classes=26
Train: 60,232   Val: 15,058
ASLClassifier(
  (net): Sequential(
    (0): Linear(in_features=63, out_features=512, bias=True)
    (1): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=512, out_features=256, bias=True)
    (5): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.3, inplace=False)
    (8): Linear(in_features=256, out_features=128, bias=True)
    (9): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): Dropout(p=0.2, inplace=False)
    (12): Linear(in_features=128, out_features=26, bias=True)
  )
)
Epoch   1/100  loss=0.6540  acc=83.68%  val_loss=0.3276  val_acc=90.92%
Epoch   2/100  loss=0.3684  acc=89.38%  val_loss=0.2845  val_acc=91.57%
Epoch   3/100  loss=0.3352  acc=